# Prepare UA-DETRAC on Colab or local Python. Run from the project repository.




In [1]:
from pathlib import Path
import os
import subprocess
import sys

# Colab mounts Drive; local runs discover the repo from the notebook working directory.
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

project_override = os.environ.get('UA_DETRAC_PROJECT')
candidates = [Path.cwd(), *Path.cwd().parents]
if project_override:
    candidates.insert(0, Path(project_override))
if IN_COLAB and not project_override:
    candidates.insert(0, Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking'))
ROOT_PATH = next(
    (path for path in candidates if (path / 'scripts/prepare_dataset.py').is_file()),
    None,
)
if ROOT_PATH is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

sys.path.insert(0, str(ROOT_PATH))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(ROOT_PATH / 'requirements.txt')],
    check=True,
)
LOCAL_RAW_PATH = Path(os.environ.get('UA_DETRAC_RAW', ROOT_PATH / 'data/raw'))
if IN_COLAB and 'UA_DETRAC_RAW' not in os.environ:
    downloaded_raw = Path('/content/ua_detrac/raw')
    if downloaded_raw.is_dir():
        LOCAL_RAW_PATH = downloaded_raw
DATA_RAW_PATH = LOCAL_RAW_PATH
WORK_ROOT = Path('/content/ua_detrac') if IN_COLAB else ROOT_PATH / 'outputs/ua_detrac'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET = Path(os.environ.get('UA_DETRAC_OUTPUT', WORK_ROOT / 'dataset_v1.1'))
print('Project:', ROOT_PATH)
print('Raw:', DATA_RAW_PATH)
print('Output:', DATASET)









Project: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking
Raw: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking\data\raw
Output: d:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking\outputs\ua_detrac\dataset_v1.1


In [ ]:
# Smoke processes one structurally valid sequence from each fixed split.
SMOKE = WORK_ROOT / 'smoke_v1.3'
result = subprocess.run(
    [sys.executable, str(ROOT_PATH / 'scripts/prepare_dataset.py'),
     '--raw', str(DATA_RAW_PATH), '--output', str(SMOKE), '--smoke'],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError('Smoke preparation failed; review the log and inventory report.')





[DATA] Inspecting actual raw XML and image frames
[DATA] Structurally valid sequences: 100/100; partial annotation: ['MVI_39051', 'MVI_39211', 'MVI_39271', 'MVI_39761', 'MVI_39781', 'MVI_39811', 'MVI_39851', 'MVI_39931', 'MVI_40152', 'MVI_40162', 'MVI_40211', 'MVI_40213', 'MVI_40743', 'MVI_40763', 'MVI_40793', 'MVI_40991', 'MVI_40992', 'MVI_63544']
[DATA] Smoke selection (complete sequences only): {'train': ['MVI_20011'], 'val': ['MVI_20012'], 'test': ['MVI_39031']}
[DATA] Parsing MVI_20011
[DATA] Parsing MVI_20012
[DATA] Parsing MVI_39031
[DATA] Validating raw images and canonical annotations
[DATA] Exporting YOLO and full-continuity MOT
[DATA] Validation PASSED: D:\ZB\Code\UIT\CS331\CS331-Vehicle-Tracking\outputs\ua_detrac\smoke_v1.3



Review the smoke dataset in notebook 03, then run the full pipeline. Existing output folders are never overwritten.




In [ ]:
# Full run includes every image, including frames without XML GT entries.
subprocess.run(
    [sys.executable, str(ROOT_PATH / 'scripts/prepare_dataset.py'),
     '--raw', str(DATA_RAW_PATH), '--output', str(DATASET)],
    check=True,
)


Create detector training subsets after full export. These contain exactly N eligible images after train_stride=5. All subsets use the same validation/test; keep checkpoints, augmentation, seed and training budget controlled.



In [2]:
# All training subsets share the same seed; validation and test stay unchanged.
for count in (500, 1000, 5000):
    subprocess.run(
        [sys.executable, str(ROOT_PATH / 'scripts/create_subset.py'),
         str(DATASET / 'detection'), '--count', str(count), '--seed', '42'],
        check=True,
    )

In [ ]:
# Package the prepared dataset and write its checksums.
ARTIFACTS = WORK_ROOT / 'artifacts'
subprocess.run(
    [sys.executable, str(ROOT_PATH / 'scripts/package_dataset.py'),
     str(DATASET), '--artifacts', str(ARTIFACTS)],
    check=True,
)

In [ ]:
import shutil

# Copy package files to the project artifacts folder.
ARTIFACT_DEST = ROOT_PATH / 'data/artifacts'
ARTIFACT_DEST.mkdir(parents=True, exist_ok=True)
for path in ARTIFACTS.iterdir():
    if path.suffix in {'.tar', '.json'}:
        shutil.copy2(path, ARTIFACT_DEST / path.name)